In [1]:
import pandas as pd

In [2]:
file_path = '../../data/benchmark/DRKG/drkg.tsv'

# 读取 TSV 文件
print("正在加载数据集...")
df = pd.read_csv(file_path, sep='\t', header=None, names=['source', 'relation', 'target'])

print(f"数据加载完成，共包含 {len(df)} 条三元组。")
# 查看前几行数据
df.head()

正在加载数据集...
数据加载完成，共包含 5874261 条三元组。


,source,relation,target
0,Gene::2157,bioarx::HumGenHumGen:Gene:Gene,Gene::2157
1,Gene::2157,bioarx::HumGenHumGen:Gene:Gene,Gene::5264
2,Gene::2157,bioarx::HumGenHumGen:Gene:Gene,Gene::2158
3,Gene::2157,bioarx::HumGenHumGen:Gene:Gene,Gene::3309
4,Gene::2157,bioarx::HumGenHumGen:Gene:Gene,Gene::28912


In [3]:
# 筛选出连接 Compound（药物）和 Disease（疾病）的所有关系类型
# 这一步是为了让你看到数据中有哪些关系，方便确认
compound_disease_rels = df[df['relation'].str.contains('Compound:Disease')]['relation'].unique()

print("数据集中包含的 '药物-疾病' 关系类型如下：")
for rel in compound_disease_rels:
    print(rel)

数据集中包含的 '药物-疾病' 关系类型如下：
DRUGBANK::treats::Compound:Disease
GNBR::T::Compound:Disease
GNBR::C::Compound:Disease
GNBR::Sa::Compound:Disease
GNBR::Pa::Compound:Disease
GNBR::Mp::Compound:Disease
GNBR::Pr::Compound:Disease
GNBR::J::Compound:Disease
Hetionet::CtD::Compound:Disease
Hetionet::CpD::Compound:Disease


In [5]:
# 定义明确表示“治疗”或“适应症”的关系列表
treatment_relation_types = [
    'Hetionet::CtD::Compound:Disease',
    'DrugBank::indication::Compound:Disease'
]

# 进行筛选
treatment_df = df[df['relation'].isin(treatment_relation_types)]

print(f"提取完成！共找到 {len(treatment_df)} 条治疗关系。")
treatment_df.head()

提取完成！共找到 755 条治疗关系。


,source,relation,target
2598941,Compound::DB00997,Hetionet::CtD::Compound:Disease,Disease::DOID:363
2598942,Compound::DB00206,Hetionet::CtD::Compound:Disease,Disease::DOID:10763
2598943,Compound::DB00960,Hetionet::CtD::Compound:Disease,Disease::DOID:10763
2598944,Compound::DB00665,Hetionet::CtD::Compound:Disease,Disease::DOID:10283
2598945,Compound::DB00290,Hetionet::CtD::Compound:Disease,Disease::DOID:2998


In [8]:
mapped_df = treatment_df.copy()
# 1. 清洗药物 ID：去掉 'Compound::' 前缀
# PrimeKG 中的药物 ID 通常就是 'DBxxxxx' 格式
mapped_df['drug_id'] = mapped_df['source'].str.replace('Compound::', '', regex=False)
# 2. 清洗疾病 ID：去掉 'Disease::' 前缀
# 得到纯净的 'DOID:xxxx'
mapped_df['doid_id'] = mapped_df['target'].str.replace('Disease::', '', regex=False)
print("ID 清洗示例：")
mapped_df[['drug_id', 'doid_id']].head()

ID 清洗示例：


,drug_id,doid_id
2598941,DB00997,DOID:363
2598942,DB00206,DOID:10763
2598943,DB00960,DOID:10763
2598944,DB00665,DOID:10283
2598945,DB00290,DOID:2998


In [9]:
import json

# 请替换为你本地 mondo.json 的路径
mondo_json_path = '../../data/benchmark/mondo.json'

print("正在加载 mondo.json (文件可能较大，请稍候)...")
with open(mondo_json_path, 'r', encoding='utf-8') as f:
    mondo_data = json.load(f)

# 构建 DOID -> MONDO 的字典
doid_to_mondo = {}

# 遍历图中的节点
# OBO Graphs JSON 结构通常在 graphs[0]['nodes'] 下
nodes = mondo_data.get('graphs', [{}])[0].get('nodes', [])

for node in nodes:
    mondo_id = node.get('id')

    # 确保是 MONDO ID
    if not mondo_id or not mondo_id.startswith('MONDO:'):
        continue

    # 查找 xrefs (外部引用)
    meta = node.get('meta', {})
    xrefs = meta.get('xrefs', [])

    for xref in xrefs:
        val = xref.get('val')
        # 如果引用的是 DOID，则记录映射关系
        if val and val.startswith('DOID:'):
            # 注意：有时一个 DOID 可能对应多个 MONDO，这里默认取其中一个
            # 或者你可以选择保留列表
            doid_to_mondo[val] = mondo_id

print(f"映射字典构建完成，包含 {len(doid_to_mondo)} 个 DOID -> MONDO 的映射关系。")

# 测试一下映射
test_doid = 'DOID:10763' # 高血压
print(f"测试映射: {test_doid} -> {doid_to_mondo.get(test_doid, '未找到')}")

正在加载 mondo.json (文件可能较大，请稍候)...
映射字典构建完成，包含 0 个 DOID -> MONDO 的映射关系。
测试映射: DOID:10763 -> 未找到
